# Day 26 — Time Series Analysis & Forecasting
## 30-Day Data Analytics Cookbook

**Objective:** Analyze observations indexed by time, identify trend and seasonality, smooth fluctuations, build a baseline forecast, and evaluate it on a chronological holdout set.

### Practical scenario
A city service team tracks monthly complaints and needs to understand demand patterns and estimate the next few months. The dataset below is synthetic and for learning only.

### Topics
- Datetime parsing and time-indexed data
- Trend and seasonality
- Monthly resampling and rolling averages
- Time-aware train/test split
- Naive baseline forecast
- MAE and RMSE
- Forecast interpretation and limitations


## 1. Import libraries

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import mean_absolute_error, mean_squared_error

print("Libraries loaded successfully")

## 2. Create a sample monthly dataset

In a real project, replace this synthetic data with an approved operational dataset. Keep one row per time period, define what the measure represents, and document missing periods.

In [ ]:
rng = np.random.default_rng(42)
dates = pd.date_range("2022-01-01", periods=48, freq="MS")
t = np.arange(len(dates))

trend = 180 + 2.2 * t
seasonality = 22 * np.sin(2 * np.pi * t / 12)
noise = rng.normal(0, 8, len(dates))

df = pd.DataFrame({
    "month": dates,
    "complaints": np.round(trend + seasonality + noise).astype(int)
})

df.head()

## 3. Prepare the time series

In [ ]:
df["month"] = pd.to_datetime(df["month"], errors="coerce")
df = df.dropna(subset=["month"]).sort_values("month")
df = df.set_index("month")

# Check duplicate timestamps and missing values.
print("Duplicate timestamps:", df.index.duplicated().sum())
print("Missing values:\n", df.isna().sum())
print("Date range:", df.index.min().date(), "to", df.index.max().date())
df.head()

## 4. Plot the observations

In [ ]:
plt.figure(figsize=(12, 4))
plt.plot(df.index, df["complaints"], marker="o", label="Monthly complaints")
plt.title("Monthly Complaint Volume")
plt.xlabel("Month")
plt.ylabel("Complaint count")
plt.legend()
plt.tight_layout()
plt.show()

**Interpretation prompts**
1. Is the overall direction increasing, decreasing, or stable?
2. Do some months repeatedly show higher or lower values?
3. Are there unusual spikes that might reflect operational events or data issues?

A plot can suggest patterns, but domain knowledge is needed to explain their causes.

## 5. Calculate a rolling average

In [ ]:
df["rolling_3_month"] = df["complaints"].rolling(window=3, min_periods=3).mean()

plt.figure(figsize=(12, 4))
plt.plot(df.index, df["complaints"], marker=".", label="Observed")
plt.plot(df.index, df["rolling_3_month"], label="3-month rolling average")
plt.title("Observed Series and Rolling Average")
plt.xlabel("Month")
plt.ylabel("Complaint count")
plt.legend()
plt.tight_layout()
plt.show()

The rolling average smooths short-term variation. It is a descriptive tool, not automatically a forecasting model. The first two values are missing because a complete three-month window is not yet available.

## 6. Split chronologically

In [ ]:
# Keep the final 6 months as a test set.
test_size = 6
train = df["complaints"].iloc[:-test_size]
test = df["complaints"].iloc[-test_size:]

print("Training period:", train.index.min().date(), "to", train.index.max().date())
print("Testing period:", test.index.min().date(), "to", test.index.max().date())

Do not randomly shuffle time-series observations for this evaluation. The model must be trained only on information that would have been available at the forecast origin.

## 7. Build a naive baseline forecast

In [ ]:
# Baseline: forecast every test month using the last training observation.
naive_forecast = pd.Series(train.iloc[-1], index=test.index, name="naive_forecast")

results = pd.DataFrame({
    "actual": test,
    "naive_forecast": naive_forecast
})
results

## 8. Evaluate forecast accuracy

In [ ]:
mae = mean_absolute_error(results["actual"], results["naive_forecast"])
rmse = np.sqrt(mean_squared_error(results["actual"], results["naive_forecast"]))

print(f"MAE:  {mae:.2f}")
print(f"RMSE: {rmse:.2f}")

- **MAE** is the average absolute forecast error in the target's units.
- **RMSE** penalizes large errors more strongly than MAE.
- Neither metric is meaningful in isolation: compare against other baselines and consider the operational cost of under- or over-forecasting.

## 9. Plot actual values versus forecast

In [ ]:
plt.figure(figsize=(12, 4))
plt.plot(train.index, train, label="Training data")
plt.plot(test.index, test, marker="o", label="Actual test data")
plt.plot(naive_forecast.index, naive_forecast, marker="x", label="Naive forecast")
plt.title("Baseline Forecast Evaluation")
plt.xlabel("Month")
plt.ylabel("Complaint count")
plt.legend()
plt.tight_layout()
plt.show()

## 10. Optional extension: seasonal naive forecast

For monthly data with annual seasonality, a seasonal-naive forecast repeats the value from the same month in the previous year. This requires enough historical data and should be evaluated on a chronological holdout.

```python
# If at least 12 training months are available:
seasonal_forecast = pd.Series(
    [train.iloc[-12 + i] for i in range(len(test))],
    index=test.index
)
```

For a test period longer than 12 months, use a robust indexed implementation rather than relying on the simple list above.

## 11. Practice exercise

1. Change the test period from 6 months to 12 months.
2. Compare a last-value naive forecast with a seasonal-naive forecast.
3. Calculate MAE and RMSE for both.
4. Try monthly data with a stronger seasonal effect.
5. Write a 3-sentence interpretation for a service manager.

### Expected analytical conclusion
A useful forecasting workflow preserves time order, establishes a simple baseline, evaluates on unseen future periods, and reports uncertainty. More complex models such as exponential smoothing, ARIMA, or Prophet should be considered only when they improve out-of-sample performance and are suitable for the data volume and pattern.

## Key takeaways

- Time-series data must be ordered by time and checked for gaps and duplicates.
- Trend and seasonality are different patterns and may coexist.
- Rolling averages help reveal smoother movement but can lag behind sudden changes.
- Chronological evaluation avoids future-data leakage.
- Always compare advanced models with a baseline and communicate forecast uncertainty.

**Next:** Day 27 — Communicating Analytical Insights / Data Storytelling (adjust to your published series sequence if needed).